In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Connect to our project's utility module
sys.path.insert(0, str(Path.cwd().parent / "src"))

from lead_scoring_utils import BASE

# Load the existing dashboard dataset
data_path = (
    BASE / "data" / "dashboard_ready"
    / "lead_scores_dashboard.xlsx"
)

df = pd.read_excel(data_path)

# Keep only the columns needed for revenue intelligence
columns = [
    "lead_id",
    "actual",
    "probability",
    "tier",
    "illustrative_commission_usd",
    "expected_commission_usd",
    "probability_rank",
    "revenue_rank"
]

df = df[columns].copy()

print("Leads loaded:", len(df))
print("\nFirst 5 records:")
print(df.head().to_string(index=False))

Leads loaded: 1386

First 5 records:
  lead_id  actual  probability tier  illustrative_commission_usd  expected_commission_usd  probability_rank  revenue_rank
LEAD-0001       1     0.178947 Cold                         3000                536.84211               700          1024
LEAD-0002       0     0.175258 Cold                         5000                876.28865               800           924
LEAD-0003       1     0.735294  Hot                         8000               5882.35280               268           298
LEAD-0004       1     0.178947 Cold                        12000               2147.36844               700           683
LEAD-0005       0     0.430556 Warm                        20000               8611.11100               512           191


In [2]:
# Calculate expected commission
df["calculated_revenue"] = (
    df["probability"]
    * df["illustrative_commission_usd"]
)

# Calculate rankings independently
df["calculated_probability_rank"] = (
    df["probability"].rank(
        ascending=False,
        method="min"
    )
)

df["calculated_revenue_rank"] = (
    df["calculated_revenue"].rank(
        ascending=False,
        method="min"
    )
)

# Verify existing calculations
print(
    "Revenue calculations match:",
    np.allclose(
        df["calculated_revenue"],
        df["expected_commission_usd"],
        atol=0.01,
        rtol=0
    )
)

print(
    "Probability ranks match:",
    (
        df["calculated_probability_rank"]
        == df["probability_rank"]
    ).sum(),
    "of", len(df)
)

print(
    "Revenue ranks match:",
    (
        df["calculated_revenue_rank"]
        == df["revenue_rank"]
    ).sum(),
    "of", len(df)
)

Revenue calculations match: True
Probability ranks match: 1386 of 1386
Revenue ranks match: 1386 of 1386


In [3]:
# Assume the sales team can contact 100 leads
capacity = 100

# Strategy A: Prioritize conversion probability
top_probability = df.nsmallest(
    capacity, "probability_rank"
)

# Strategy B: Prioritize expected commission
top_revenue = df.nsmallest(
    capacity, "revenue_rank"
)

# Compare the strategies
comparison = pd.DataFrame({
    "Strategy": [
        "Probability-based",
        "Revenue-based"
    ],
    "Leads Contacted": [
        len(top_probability),
        len(top_revenue)
    ],
    "Expected Commission (USD)": [
        top_probability["calculated_revenue"].sum(),
        top_revenue["calculated_revenue"].sum()
    ],
    "Actual Conversions": [
        top_probability["actual"].sum(),
        top_revenue["actual"].sum()
    ]
})

print(comparison.to_string(index=False))

         Strategy  Leads Contacted  Expected Commission (USD)  Actual Conversions
Probability-based              100                1010686.908                  94
    Revenue-based              100                1529288.476                  85


In [4]:
# Identify leads selected by each strategy
probability_ids = set(top_probability["lead_id"])
revenue_ids = set(top_revenue["lead_id"])

# Compare the two selections
common_leads = probability_ids & revenue_ids

print("Leads selected by both strategies:", len(common_leads))
print("Only probability-based:", len(probability_ids - revenue_ids))
print("Only revenue-based:", len(revenue_ids - probability_ids))

# Show the five highest-ranked revenue opportunities
print("\nTop 5 revenue-priority leads:")

print(
    top_revenue[
        [
            "lead_id",
            "probability",
            "illustrative_commission_usd",
            "calculated_revenue",
            "probability_rank",
            "revenue_rank"
        ]
    ]
    .head(5)
    .to_string(index=False)
)

Leads selected by both strategies: 35
Only probability-based: 65
Only revenue-based: 65

Top 5 revenue-priority leads:
  lead_id  probability  illustrative_commission_usd  calculated_revenue  probability_rank  revenue_rank
LEAD-0670     1.000000                        20000            20000.00                 1             1
LEAD-0015     0.959596                        20000            19191.92                 8             2
LEAD-0205     0.959596                        20000            19191.92                 8             2
LEAD-0220     0.959596                        20000            19191.92                 8             2
LEAD-0240     0.959596                        20000            19191.92                 8             2


In [5]:
def prioritize_leads(data, capacity=100, strategy="revenue"):
    """
    Select leads based on the chosen business objective.
    """

    if strategy == "revenue":
        ranking_column = "calculated_revenue"

    elif strategy == "probability":
        ranking_column = "probability"

    else:
        raise ValueError(
            "Strategy must be 'revenue' or 'probability'"
        )

    prioritized = (
        data.sort_values(
            by=ranking_column,
            ascending=False,
            kind="stable"
        )
        .head(capacity)
        .copy()
    )

    return prioritized


# Test our function
priority_list = prioritize_leads(
    df,
    capacity=10,
    strategy="revenue"
)

print(
    priority_list[
        [
            "lead_id",
            "probability",
            "calculated_revenue",
            "tier"
        ]
    ].to_string(index=False)
)

  lead_id  probability  calculated_revenue tier
LEAD-0670     1.000000            20000.00  Hot
LEAD-0015     0.959596            19191.92  Hot
LEAD-0205     0.959596            19191.92  Hot
LEAD-0220     0.959596            19191.92  Hot
LEAD-0240     0.959596            19191.92  Hot
LEAD-0245     0.959596            19191.92  Hot
LEAD-0280     0.959596            19191.92  Hot
LEAD-0290     0.959596            19191.92  Hot
LEAD-0465     0.959596            19191.92  Hot
LEAD-0540     0.959596            19191.92  Hot


In [6]:
import joblib
from lead_scoring_utils import prepare, tiers

# Load the verified model
model_path = (
    BASE / "models"
    / "lead_scoring_pipeline_v2.joblib"
)

bundle = joblib.load(model_path)

def score_new_leads(new_leads):
    """
    Score incoming leads using the saved model.
    """

    # Prepare the required model features
    X = prepare(new_leads)

    # Generate raw model probabilities
    raw_probabilities = (
        bundle["pipeline"].predict_proba(X)[:, 1]
    )

    # Apply probability calibration
    probabilities = bundle["calibrator"].predict(
        raw_probabilities
    )

    # Build scored output
    results = new_leads.copy()

    results["probability"] = probabilities
    results["tier"] = tiers(probabilities)

    return results


print("New-lead scoring function created successfully.")

New-lead scoring function created successfully.


In [7]:
from lead_scoring_utils import load_split

# Load our existing held-out test data
train, val, test = load_split()

# Simulate three incoming leads
incoming_leads = test.head(3).copy()

# Remove the historical outcome
incoming_leads = incoming_leads.drop(
    columns=["Converted"]
)

# Assign demonstration lead IDs
incoming_leads["lead_id"] = [
    "NEW-001",
    "NEW-002",
    "NEW-003"
]

# Score the incoming leads
scored_leads = score_new_leads(incoming_leads)

# Display the results
print(
    scored_leads[
        ["lead_id", "probability", "tier"]
    ].to_string(index=False)
)

lead_id  probability tier
NEW-001     0.178947 Cold
NEW-002     0.175258 Cold
NEW-003     0.735294  Hot


In [8]:
# Demonstration commission values
commission_values = [3000, 5000, 8000]

# Add illustrative commission to the scored leads
scored_leads["illustrative_commission_usd"] = commission_values

# Calculate expected commission
scored_leads["calculated_revenue"] = (
    scored_leads["probability"]
    * scored_leads["illustrative_commission_usd"]
)

# Prioritize all three leads by expected commission
sales_priority_list = prioritize_leads(
    scored_leads,
    capacity=3,
    strategy="revenue"
).reset_index(drop=True)

# Assign sequential action priorities
sales_priority_list["action_priority"] = (
    np.arange(1, len(sales_priority_list) + 1)
)

# Display the actionable sales list
print(
    sales_priority_list[
        [
            "action_priority",
            "lead_id",
            "probability",
            "tier",
            "illustrative_commission_usd",
            "calculated_revenue"
        ]
    ].to_string(index=False)
)

 action_priority lead_id  probability tier  illustrative_commission_usd  calculated_revenue
               1 NEW-003     0.735294  Hot                         8000         5882.352829
               2 NEW-002     0.175258 Cold                         5000          876.288638
               3 NEW-001     0.178947 Cold                         3000          536.842123
